In [1]:
import numpy as np
import pandas as pd 
import os

### A content-based recommendation system recommends items to users based on the content or characteristics of the items. This type of recommendation system focuses on understanding the properties of items and learning user preferences from the items they have interacted with in the past.

How Does it Work?

1. Feature Extraction: Extract relevant features from the items. For example, in a book recommendation system, features could include title, author, and category

2. User Profile: Create a user profile based on their interactions with items. This profile is essentially a summary of the features of items the user has liked or interacted with in the past.

3. Recommendation: Calculate the similarity between the user profile and each item's features. Items that are most similar to the user profile are recommended.

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

content_books = pd.read_csv('../data/original/Books.csv')
content_users = pd.read_csv('../data/original/Users.csv')
content_ratings = pd.read_csv('../data/original/Ratings.csv')


C:\Users\Mukand\AppData\Local\Temp\ipykernel_18216\4233080962.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  content_books = pd.read_csv('../data/original/Books.csv')


In [21]:
content_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   User-ID      1048575 non-null  int64 
 1   ISBN         1048575 non-null  object
 2   Book-Rating  1048575 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 24.0+ MB


In [5]:
content_ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,155061224,5
2,276727,446520802,0
3,276729,052165615X,3
4,276729,521795028,6


In [6]:
print("Null values in books data: \n", content_books.isnull().sum())

Null values in books data: 
 ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64


In [7]:
# Dropping unnecessary columns
content_books.drop(['Image-URL-S', 'Image-URL-M', 'Image-URL-L', 'Year-Of-Publication'], axis=1, inplace=True)

content_users.drop(['Location', 'Age'], axis=1, inplace=True)

In [8]:
# Merge ratings data with cleaned books and users data
content = pd.merge(content_ratings, content_books, on='ISBN')
content = pd.merge(content, content_users, on='User-ID')

In [9]:
content.shape

(941113, 6)

In [10]:
content.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Publisher
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,Ballantine Books
1,276726,155061224,5,Rites of Passage,Judith Rae,Heinle
2,276727,446520802,0,The Notebook,Nicholas Sparks,Warner Books
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,Cambridge University Press
4,276729,521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,Cambridge University Press


In [11]:
content.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 941113 entries, 0 to 941112
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   User-ID      941113 non-null  int64 
 1   ISBN         941113 non-null  object
 2   Book-Rating  941113 non-null  int64 
 3   Book-Title   941113 non-null  object
 4   Book-Author  941111 non-null  object
 5   Publisher    941111 non-null  object
dtypes: int64(2), object(4)
memory usage: 43.1+ MB


In [12]:
# Concatenate relevant columns to create feature
content['Features'] = content['Book-Title'] + ', ' + content['Book-Author'] + ', ' + content['Publisher']

In [13]:
content.head()

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Publisher,Features
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,Ballantine Books,"Flesh Tones: A Novel, M. J. Rose, Ballantine B..."
1,276726,155061224,5,Rites of Passage,Judith Rae,Heinle,"Rites of Passage, Judith Rae, Heinle"
2,276727,446520802,0,The Notebook,Nicholas Sparks,Warner Books,"The Notebook, Nicholas Sparks, Warner Books"
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,Cambridge University Press,"Help!: Level 1, Philip Prowse, Cambridge Unive..."
4,276729,521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,Cambridge University Press,The Amsterdam Connection : Level 4 (Cambridge ...


In [14]:
content.drop(['Book-Author', 'Publisher', 'User-ID', 'Book-Rating'], axis=1, inplace=True)

In [15]:
content.isnull().sum()

ISBN          0
Book-Title    0
Features      4
dtype: int64

In [16]:
# Drop rows with NaN values in columns
content.dropna(subset=['Features'], inplace=True)

In [17]:
content['Features'] = content['Features'].apply(lambda x : str(x).lower())
content

,ISBN,Book-Title,Features
0,034545104X,Flesh Tones: A Novel,"flesh tones: a novel, m. j. rose, ballantine b..."
1,155061224,Rites of Passage,"rites of passage, judith rae, heinle"
2,446520802,The Notebook,"the notebook, nicholas sparks, warner books"
3,052165615X,Help!: Level 1,"help!: level 1, philip prowse, cambridge unive..."
4,521795028,The Amsterdam Connection : Level 4 (Cambridge ...,the amsterdam connection : level 4 (cambridge ...
...,...,...,...
941108,451410777,Sleep Tight (Onyx Book),"sleep tight (onyx book), anne frasier, onyx books"
941109,452264464,Beloved (Plume Contemporary Fiction),"beloved (plume contemporary fiction), toni mor..."
941110,048623715X,Glamorous Movie Stars of the Thirties: Paper D...,glamorous movie stars of the thirties: paper d...
941111,486256588,Schiaparelli Fashion Review: Paper Dolls in Fu...,schiaparelli fashion review: paper dolls in fu...


In [18]:
content.head()

,ISBN,Book-Title,Features
0,034545104X,Flesh Tones: A Novel,"flesh tones: a novel, m. j. rose, ballantine b..."
1,155061224,Rites of Passage,"rites of passage, judith rae, heinle"
2,446520802,The Notebook,"the notebook, nicholas sparks, warner books"
3,052165615X,Help!: Level 1,"help!: level 1, philip prowse, cambridge unive..."
4,521795028,The Amsterdam Connection : Level 4 (Cambridge ...,the amsterdam connection : level 4 (cambridge ...


In [24]:
sample_size = 2000

content_cv = content.sample(n=sample_size, random_state=42)

unique_book_count = content_cv['Book-Title'].nunique()
print("Number of unique books:", unique_book_count)
content_cv.shape

Number of unique books: 1902


(2000, 3)

### using count_vectorizer

In [26]:
content_cv.head()

,ISBN,Book-Title,Features
472846,375412115,Our Lady of the Forest,"our lady of the forest, david guterson, knopf"
707310,425180433,Ten Thousand Islands,"ten thousand islands, randy wayne white, prime..."
837480,380762250,Flute Song Magic,"flute song magic, andrea shettle, harpercollin..."
368485,1931696624,Dayspring Dawning,"dayspring dawning, jeanine berry, novelbooks"
214774,590228544,The Terrorist (Point),"the terrorist (point), caroline b. cooney, sch..."


In [27]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=1000, stop_words='english')
cv.fit_transform(content_cv['Features']).toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [28]:
profiles = cv.fit_transform(content_cv['Features']).toarray()
profiles.shape

(2000, 1000)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(profiles)
print("Cosine similarity shape:", cosine_sim.shape)

Cosine similarity shape: (2000, 2000)


In [50]:
def recommend_cv(book_title, top_n=5):
    try:
        index = content_cv[content_cv['Book-Title'] == book_title].index[0]
        similar_books = sorted(enumerate(cosine_sim[index]), key=lambda x: x[1], reverse=True)[1:top_n+1]
        print(f"Top {top_n} recommended books for '{book_title}':")
        for i in similar_books:
            print(content_cv.iloc[i[0]]['Book-Title'], "| Similarity:", round(i[1], 3))
    except IndexError:
        print("Book not found in sample.")
content_cv = content.sample(n=sample_size, random_state=42).reset_index(drop=True)

In [51]:
def recommend_cv(book):
    index = np.where(content_cv['Book-Title'] == book)[0][0]
    similar_books = sorted(enumerate(cosine_sim[index]), key=lambda x: x[1], reverse=True)[1:10]

    print("\nTop 5 recommended movies based on your preferences:")
    for i in similar_books:
#         print(content_cv['Book-Title'][i[0]])
        print(content['Book-Title'][i[0]])

In [52]:
content_cv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ISBN        2000 non-null   object
 1   Book-Title  2000 non-null   object
 2   Features    2000 non-null   object
dtypes: object(3)
memory usage: 47.0+ KB


In [53]:
print(content_cv['Book-Title'].head(10))

0                 Our Lady of the Forest
1                   Ten Thousand Islands
2                       Flute Song Magic
3                      Dayspring Dawning
4                  The Terrorist (Point)
5                      The Golden People
6    The Knife Thrower and Other Stories
7                       Call of the Wild
8                               The Mask
9                      Der Bibliothekar.
Name: Book-Title, dtype: object


## Recommendations

In [54]:
recommend_cv('Ten Thousand Islands')


Top 5 recommended movies based on your preferences:
Was wÃ?Â¤re gewesen, wenn? Wendepunkte der Weltgeschichte.
Chocolat
Diary of a Provincial Lady (Provincial Lady)
Wide Sargasso Sea
Christmas Bonus, Strings Attached
The Power That Preserves (The Chronicles of Thomas Covenant the Unbeliever, Book 3)
Girl, Interrupted
If You Were a Firefighter (If You Were a...)
Charmed Circle


In [ ]:
from tabulate import tabulate
def recommend_book_cv(book_title, top_n=5):
    try:
        index = content_cv[content_cv['Book-Title'] == book_title].index[0]
        similar_books = sorted(enumerate(cosine_sim[index]), key=lambda x: x[1], reverse=True)[1:top_n+1]
        table_data = []
        for i in similar_books:
            recommended_index = i[0]
            recommended_title = content_cv.iloc[recommended_index]['Book-Title']
            similarity_score = i[1]
            table_data.append([recommended_index, recommended_title, round(similarity_score, 6)])
        print(tabulate(table_data, headers=['Index', 'Recommended Book', 'Similarity Score'], tablefmt='grid'))
    except IndexError:
        print("Book not found in sample.")

In [ ]:
recommend_book_cv('Dayspring Dawning')
recommend_book_cv('A Wrinkle in Time')
recommend_book_cv('Ten Thousand Islands')

+---------+----------------------------------------------------------------+--------------------+
|   Index | Recommended Book                                               |   Similarity Score |
+=========+================================================================+====================+
|       1 | Rites of Passage                                               |                  0 |
+---------+----------------------------------------------------------------+--------------------+
|       2 | The Notebook                                                   |                  0 |
+---------+----------------------------------------------------------------+--------------------+
|       3 | Help!: Level 1                                                 |                  0 |
+---------+----------------------------------------------------------------+--------------------+
|       4 | The Amsterdam Connection : Level 4 (Cambridge English Readers) |                  0 |
+---------+---------

In [60]:
duplicate_counts = content_cv['Book-Title'].value_counts()
duplicate_books = duplicate_counts[duplicate_counts > 1]
total_duplicate_count = duplicate_books.sum()
print("Total count of duplicate books:", total_duplicate_count)


for book_title, count in duplicate_books.items():
    indices = content_cv.index[content_cv['Book-Title'] == book_title].tolist()
    print(f"Book Title: {book_title} (Count: {count}) - Indices: {indices}")

Total count of duplicate books: 184
Book Title: Angels &amp; Demons (Count: 4) - Indices: [382, 528, 1778, 1879]
Book Title: The Nanny Diaries: A Novel (Count: 4) - Indices: [653, 682, 1642, 1851]
Book Title: Life of Pi (Count: 3) - Indices: [363, 1106, 1489]
Book Title: The Beach House (Count: 3) - Indices: [802, 822, 825]
Book Title: The Da Vinci Code (Count: 3) - Indices: [409, 1552, 1654]
Book Title: Tuesdays with Morrie: An Old Man, a Young Man, and Life's Greatest Lesson (Count: 3) - Indices: [195, 540, 1431]
Book Title: The Rescue (Count: 3) - Indices: [434, 753, 815]
Book Title: Night Sins (Count: 3) - Indices: [476, 1275, 1816]
Book Title: Flesh and Blood (Count: 3) - Indices: [300, 1335, 1408]
Book Title: The Mists of Avalon (Count: 3) - Indices: [518, 808, 1356]
Book Title: Nightmares &amp; Dreamscapes (Count: 2) - Indices: [878, 903]
Book Title: Harry Potter Schoolbooks: Quidditch Through the Ages and Fantastic Beasts and Where to Find Them (Count: 2) - Indices: [134, 513]


In [61]:
unique_book_count = content_cv['Book-Title'].nunique()
print("Number of unique books:", unique_book_count)
content_cv.shape

Number of unique books: 1902


(2000, 3)

### Should use Count Vectorizer: 
to convert the textual features into numerical vectors. It simply counts the frequency of each word in the text data and creates a matrix where each row represents a book in my case) and each column represents a unique word in the corpus. 
If not concerned about down-weighting the presence of certain words (such as Book-Title, Book-Author, and Publisher) based on their frequency across all books (documents), CountVectorizer is more suitable than TF-IDF.

### Reasoning: 
In the context of content-based recommendation systems for books, I want to capture the presence of specific words (such as Book-Title, Book-Author, and Publisher) in each book's features. CountVectorizer does exactly that by representing each book's textual features as a frequency count of words, which helps in identifying similarities between books based on their textual content. 
As I'm interested in treating all words equally without considering their rarity across all books, TF-IDF's IDF component is not necessary in this case.